# Validation KUL20 new

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

In [ ]:
import os
import sys
import glob
import scipy
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# PlatoSim libraries
import platosim.plot      as pt
import platosim.utilities as ut
from platosim.simfile      import SimFile
from platosim.simulation   import Simulation
from platosim.lightcurve   import LightCurve
from platosim.matplotlibrc import setup_paper
setup_paper()

In [ ]:
path = '/lhome/nicholas/software/workdir/kul20_new'

---
## Camera level
---

### NSR from single quarter light curve

In [ ]:
lcs = LightCurve(f'{path}/nsr/000000001', mode='multi')

In [ ]:
lc = LightCurve(lcs.files("hdf5")[0])

In [ ]:
lc.get_nsr(), lc.get_nsr_new()

In [ ]:
fig, ax = lc.plot(time_unit="h", flux_unit='ppm', binsize=1, alpha=0.5, figsize=(8,4));

### NSR limit from PINE

PINE (systematic) jitter noise:
- 25s BOL (EOL): 108 ppm
- 01h BOL (EOL): 9 ppm

In [ ]:
NSR_jitter0 = ut.getJitterNoiseLimitNSR(0.054, tdur=25,   camType='normal')
NSR_jitter1 = ut.getJitterNoiseLimitNSR(0.054, tdur=3600, camType='normal')
NSR_jitter2 = ut.getJitterNoiseLimitNSR(0.040, tdur=3600, camType='normal')
NSR_jitter0, NSR_jitter1, NSR_jitter2

PINE (random) photon noise 1h:
- 01 N-CAM (BOL, EOL) = (388.5, 490.2) ppm
- 24 N-CAM (BOL, EOL) = (44.3, 49.3) ppm

In [ ]:
NSR_photon0 = ut.getPhotonNoiseLimitNSR(11, passband='P', camType='normal', ncam=1,  ntra=1, tdur=3600)
NSR_photon1 = ut.getPhotonNoiseLimitNSR(11, passband='P', camType='normal', ncam=24, ntra=1, tdur=3600)
NSR_photon0, NSR_photon1

Sky and read noise per 1h:
-  1 N-CAM BOL (EOL): () ppm
- 24 N-CAM BOL (EOL): () ppm

### NSR for all cameras

In [ ]:
ofile_ncam = f'{path}/nsr_per_ncam_Q15.ftr'

# Compute NSR per camera
lcs = LightCurve(f'{path}/nsr', 'multi')
lcs.get_nsr_per_camera(ofile_ncam, suffix='hdf5', quarter=15)

# Load results and add Vmag
df = pd.read_feather(ofile_ncam)
df['Pmag'] = df.mag
df['Teff'] = np.ones_like(df.mag) * 6000
df['Vmag'] = ut.passbandConversionV2P(df.mag, df.Teff, inverse=True, method='fialho')
df.head()

In [ ]:
# Order after contaminants
df['mag'] = df.Pmag
col = 'rOA'
df = df.sort_values(by=[col], ascending=False)
# df = df[df.SPR < 0.06]

# Plot all data together
fig, ax = pt.plotNSRvsMagnitude(df, 
                                column=col, 
                                passband='P', 
                                show_ncam_noise_limits=1, 
                                show_saturation_limits=True,
                                legend=True, 
                                figsize=(10, 7))
# Settings
ax.set_title('Noise budget at camera level for Q15')
ax.set_xlim(6, 14)
ax.set_ylim(5, 1000)
# ax.set_xlim(8.3, 11.1)
# ax.set_ylim(50, 400)
plt.legend(loc='lower right');

---
## NSR at mission level
---

### Test NSR estimate

In [ ]:
lcs = LightCurve(f'{path}/nsr/000000001', mode='multi')

In [ ]:
fig, ax = lcs.plot_multi(suffix='hdf5', alpha=1);

In [ ]:
lc = lcs.merge(suffix='hdf5', flux_normalise=True, flux_group_mean=True)

In [ ]:
lc.stat_sim_table().iloc[0]

In [ ]:
fig, ax = lc.plot(time_unit="h", flux_unit='ppm', binsize=1, alpha=0.5, figsize=(8,4));

In [ ]:
lc.get_nsr(), lc.get_nsr_new(), lc.bin().std().flux * 1e6

In [ ]:
lc.plot_frequency_performance(figsize=(8,6));

### For all stars

In [ ]:
# Filename
ofile_star = f'{path}/nsr_per_star_Q15.ftr'

# Compute NSR per star
lcs = LightCurve(f'{path}/nsr', 'multi')
lcs.get_nsr_per_star(ofile_star, suffix='hdf5', quarter=15);

# Load results and add Vmag
df = pd.read_feather(ofile_star)
df['Pmag'] = df.mag
df['Teff'] = np.ones_like(df.shape[0]) * 6000
df['Vmag'] = ut.passbandConversionV2P(df.mag, df.Teff, inverse=True, method='fialho')
df.head()

In [ ]:
# df = pd.read_feather(ofile_ncam)
# ids = df.ID.unique()
# NSR = np.zeros(len(ids))
# for i in ids:
#     dx = df[df.ID == i]
#     NSR[i-1] = dx.NSR.mean() / np.sqrt(dx.shape[0])
# df['NSR'] = NSR

In [ ]:
# Order after contaminants
col = 'ncam'
df['mag'] = df.Vmag
df = df.sort_values(by=[col], ascending=False)

# Plot all data together
fig, ax = pt.plotNSRvsMagnitude(df, 
                                column=col, 
                                passband='V', 
                                residuals="multi",
                                show_targets=True,
                                show_ncam_noise_limits=24, 
                                figsize=(10, 7))
ax.set_title('Noise budget at mission level for Q15')
ax.set_xlim(8, 11.5)
ax.set_ylim(6, 110)
# ax.set_xlim(6, 14)
# ax.set_ylim(5, 500)
plt.legend();

In [ ]:
df[(df.ncam == 24) & (df.mag > 10.99) & (df.mag < 11.01)]

In [ ]:
ut.passbandConversionV2P(10.72, 6000, inverse=True)

### Test NSR estimate

In [ ]:
lcs = LightCurve(f'{path}/run_long', mode='multi')

In [ ]:
# fig, ax = lcs.plot_multi(suffix='hdf5', alpha=1);

In [ ]:
lc = lcs.merge(suffix='hdf5', 
               detrend=True,
               flux_normalise=False, 
               flux_group_mean=True)

In [ ]:
# lc.stat_sim_table().iloc[0]

In [ ]:
fig, ax = lc.plot(time_unit="d", flux_unit='ppm', binsize=1, alpha=0.5, figsize=(8,4));

In [ ]:
lc.get_nsr()

In [ ]:
lc.plot_frequency_performance(figsize=(9,5));